In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_URI= "sqlite:////home/nasia/wine-innovation-engine/notebooks/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_URI)

client=MlflowClient()

exp=mlflow.get_experiment_by_name("phase5_metabolomics")
runs=client.search_runs(experiment_ids=[exp.experiment_id],order_by=["start_time DESC"],max_results=1)
RUN_ID=runs[0].info.run_id

print("Using run:",RUN_ID)




In [ ]:
#lets see what artifacts we have

for f in client.list_artifacts(RUN_ID):
    print(f.path,f.file_size,"bytes")

In [ ]:
vip_path = client.download_artifacts(RUN_ID, "phase5_vip_scores.csv", dst_path="../results/tables/")
xscaled_path = client.download_artifacts(RUN_ID, "phase5_X_scaled.csv", dst_path="../results/tables/")

vip_table = pd.read_csv(vip_path, index_col=0)
X_scaled = pd.read_csv(xscaled_path, index_col=0)

print("VIP table:", vip_table.shape)
print("X_scaled:", X_scaled.shape)

vip_table.head()

In [ ]:
X_scaled.head()

In [ ]:
#lets put them in families 
chemical_families = {
    'palmitic acid': 'fatty_acid', 'stearic acid': 'fatty_acid',
    'arachidic acid': 'fatty_acid', 'heptadecanoic acid': 'fatty_acid',
    'behenic acid': 'fatty_acid', 'oleic acid': 'fatty_acid',
    '1-hexadecanol': 'fatty_alcohol', 'octadecanol': 'fatty_alcohol',
    'tartaric acid': 'organic_acid', 'isocitric acid': 'organic_acid',
    '2-ketoisocaproic acid': 'organic_acid', 'maleic acid': 'organic_acid',
    'benzoic acid': 'organic_acid', 'threonic acid': 'organic_acid',
    '3-hydroxy-3-methylglutaric acid': 'organic_acid',
    'ribose': 'sugar_polyol', 'inositol myo-': 'sugar_polyol',
    'sophorose': 'sugar_polyol', 'xylitol': 'sugar_polyol',
    'fucose': 'sugar_polyol', 'glyceric acid': 'sugar_polyol',
    '4-hydroxyproline': 'amino_acid_derivative',
    'suberyl glycine': 'amino_acid_derivative',
    'uracil': 'nucleobase',
}

In [ ]:
#the different chemical families consumed

processes = {
    'fermentation_progress': {
        'consumes': ['fatty_acid', 'fatty_alcohol'],   
        'note': 'Palmitic/stearic decline through fermentation (Skogerson 2009, refs 31-32)'
    },
    'yeast_autolysis': {
        'produces': ['sugar_polyol'],                  
        'note': 'Ribose, myo-inositol appear as yeast dies (Phase 1 Xt state)'
    },
    'grape_ripeness': {
        'produces': ['amino_acid_derivative'],           
        'note': 'Structural amino acids from grape phenolic maturity'
    },
}

In [ ]:
#lets build the nodes 

def build_metabolomics_graph(vip_table,chemical_families,processes,top_n=20,corr_threshold=0.7,X_scaled=None): #we put none ath the scaled because if we dont put anything it will continue to the next
    G=nx.DiGraph()
    #central node
    G.add_node("body_score",type="outcome",label="Wine Body Score")#the label will be use to show up in the screen and the type for the coloring
    #family of nodes
    for fam in set(chemical_families.values()): #we use set to remove duplicates
        G.add_node(fam,type="family",label=fam.replace('_',' ').title())#fatty_acid->Fatty acid

    #nodes of process
    for proc in processes:
        G.add_node(proc,type="process",label=proc.replace('_',' ').title())

    #the metabolites as nodes
    top=vip_table.head(top_n)

    for met_raw,row in top.iterrows(): #iterrows gives two things -> index and row
        met=met_raw.strip()  
        G.add_node(met,type="metabolite",vip=float(row['VIP']),direction=row['direction'])

        relation="increases" if row['direction']=='higher body' else 'decreases'
        G.add_edge(met,"body_score",relation=relation,weight=float(row['VIP']))
        #we want thicker line for the most important metabolits
        
        #connection to the family 
        fam=chemical_families.get(met)#get finds the clean met,if it does not exist it returns None
        if fam:
            G.add_edge(met,fam,relation="belong_to")

        #edge process to family
        for proc,info in processes.items():
            for fam in info.get('consumes',[]):#the info is the inside dictionary
                if fam in G:#if it already exists in the graph
                    G.add_edge(proc,fam,relation='consumes',note=info['note'])


        #if we have X_scaled
    if X_scaled is not None:
        met_cols = [m for m in top.index if m in X_scaled.columns]
        corr = X_scaled[met_cols].corr()
        for i, m1 in enumerate(met_cols):
            for m2 in met_cols[i+1:]:
                r = corr.loc[m1, m2]
                if abs(r) >= corr_threshold:
                    G.add_edge(m1.strip(), m2.strip(), relation="correlates_with", weight=float(r))
    
    return G

G = build_metabolomics_graph(vip_table, chemical_families, processes, top_n=20,corr_threshold=0.85,X_scaled=X_scaled)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Node types: {pd.Series([d['type'] for _, d in G.nodes(data=True)]).value_counts().to_dict()}")









In [ ]:
net = Network(height="750px", width="100%", directed=True, notebook=True, bgcolor="#1e1e1e", font_color="white",  cdn_resources='remote')
net.from_nx(G)

type_colors = {
    'outcome': '#ffd700',
    'metabolite': '#4ECDC4',
    'family': '#FF6B6B',
    'process': '#95E1D3',
}

for node in net.nodes:
    node_type = G.nodes[node['id']].get('type', 'metabolite')
    node['color'] = type_colors.get(node_type, '#cccccc')
    
    if node_type == 'metabolite':
        vip = G.nodes[node['id']].get('vip', 1)
        node['size'] = 15 + vip * 8            # μεγαλύτερος κόμβος = υψηλότερο VIP
        node['title'] = f"VIP: {vip:.2f}\nDirection: {G.nodes[node['id']].get('direction')}"
    elif node_type == 'outcome':
        node['size'] = 40
    else:
        node['size'] = 25

for edge in net.edges:
    rel = edge.get('relation', '')
    w = edge.get('weight', 1)
    edge['width'] = abs(w) * 2 if rel == 'correlates_with' else 2
    edge['title'] = rel
    if rel == 'decreases':
        edge['color'] = '#d62728'
    elif rel == 'increases':
        edge['color'] = '#2ca02c'
    elif rel == 'correlates_with':
        edge['color'] = '#888888'
        edge['dashes'] = True

net.toggle_physics(True)
net.show_buttons(filter_=['physics'])
net.show("../results/phase5_knowledge_graph.html")

In [ ]:
for proc in processes:
    G.add_node(proc,"->successors:",list(G.successors(proc)))